## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (10, 5)

SEED = 42

In [ ]:
# Cambiar el directorio de trabajo a la raíz del proyecto para que los archivos se guarden fuera de la carpeta Notebooks
import os
if os.getcwd().endswith('Notebooks'):
    os.chdir('..')
print(f"Directorio de trabajo actual: {os.getcwd()}")

## 2. Carga del dataset

Se utiliza la fuente directa del repositorio de IBM en GitHub, que contiene el mismo archivo CSV disponible en Kaggle (`blastchar/telco-customer-churn`). Esto permite reproducir el notebook sin necesidad de cuenta Kaggle.

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df_raw = pd.read_csv(URL)

print(f'Dimensiones del dataset: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')
df_raw.head()

## 3. Exploración inicial

### 3.1 Tipos de datos y primeras observaciones

In [ ]:
print('=== Tipos de datos ===')
print(df_raw.dtypes)

In [ ]:
df_raw.describe(include='all').T

**Interpretación:**
- `tenure` va de 0 a 72 meses (hasta 6 años de antigüedad).
- `MonthlyCharges` oscila entre ~18 y ~119 USD/mes.
- `TotalCharges` es tipo *object*, lo que indica valores no numéricos que requieren limpieza.
- Variables como `gender`, `Partner`, `Dependents` son binarias codificadas como texto.
- `SeniorCitizen` ya es numérica (0/1).

### 3.2 Variable objetivo: distribución de Churn

In [ ]:
churn_counts = df_raw['Churn'].value_counts()
churn_pct    = df_raw['Churn'].value_counts(normalize=True) * 100

print('Distribución de Churn:')
print(pd.DataFrame({'Cantidad': churn_counts, 'Porcentaje (%)': churn_pct.round(2)}))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Barras
sns.barplot(x=churn_counts.index, y=churn_counts.values, palette=['#4C8BF5','#E74C3C'], ax=axes[0])
axes[0].set_title('Distribución absoluta de Churn')
axes[0].set_ylabel('Número de clientes')
axes[0].set_xlabel('')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

# Pie
axes[1].pie(churn_counts.values, labels=churn_counts.index,
            autopct='%1.1f%%', colors=['#4C8BF5','#E74C3C'],
            startangle=140, wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Proporción de Churn')

plt.suptitle('Variable objetivo: Churn', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Interpretación:**  
El dataset presenta **desbalance de clases**: aproximadamente el 73% de los clientes no abandonó el servicio (`No`) frente al 27% que sí lo hizo (`Yes`). Este desbalance debe considerarse durante el modelado — se usará el parámetro `class_weight='balanced'` (Random Forest) o `scale_pos_weight` (XGBoost) para compensarlo.

## 4. Análisis de variables numéricas

### 4.1 Distribuciones univariadas

In [ ]:
# TotalCharges debe convertirse antes de graficar
df_raw['TotalCharges'] = pd.to_numeric(df_raw['TotalCharges'], errors='coerce')

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for i, col in enumerate(num_cols):
    # Histograma + KDE
    sns.histplot(df_raw[col].dropna(), kde=True, ax=axes[0, i],
                 color='steelblue', edgecolor='white', linewidth=0.4)
    axes[0, i].set_title(f'Distribución: {col}')
    axes[0, i].set_xlabel('')

    # Boxplot por Churn
    sns.boxplot(data=df_raw, x='Churn', y=col, palette=['#4C8BF5','#E74C3C'], ax=axes[1, i])
    axes[1, i].set_title(f'{col} por Churn')

plt.suptitle('Variables numéricas: distribución y relación con Churn', fontsize=13)
plt.tight_layout()
plt.show()

**Interpretación:**
- **`tenure`**: distribución bimodal. Los clientes que abandonan tienden a tener menor antigüedad — muchos se van en los primeros meses.
- **`MonthlyCharges`**: los clientes con churn tienen medianas de cargo mensual más altas, lo que sugiere que los planes más costosos generan mayor insatisfacción.
- **`TotalCharges`**: fuertemente correlacionado con `tenure`. Los clientes que abandonan acumulan menos cargos totales (lógico dado su corto tiempo de servicio).

### 4.2 Matriz de correlación

In [ ]:
corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
corr = df_raw[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, ax=ax, linewidths=0.5,
            annot_kws={'size': 12})
ax.set_title('Correlación entre variables numéricas')
plt.tight_layout()
plt.show()

**Interpretación:**  
`tenure` y `TotalCharges` presentan correlación alta (~0.83), lo que es esperable: a mayor tiempo como cliente, mayor facturación acumulada. Esto introduce **multicolinealidad leve** que no es problemática para modelos basados en árboles, pero debe tenerse en cuenta si se compara con regresión logística.

## 5. Análisis de variables categóricas

In [ ]:
cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
            'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
            'PaperlessBilling', 'PaymentMethod']

n_cols = 3
n_rows = int(np.ceil(len(cat_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.flatten()

# More distinguishable 4-color palette from blue to red
# Using clearer progression with better contrast
""""
colors_4 = [
    '#4C8BF5',  # Original blue (lowest churn)
    '#2C6FD1',  # Darker blue (more contrast)
    '#D43F29',  # Darker red/orange
    '#E74C3C'   # Original red (highest churn)
]
"""
# Alternative 4-color palette (more gradient-like)
colors_4 = [
   '#4C8BF5',  # Pure blue
   '#5B8EC4',  # Blue-purple
   '#C55E4C',  # Red-orange  
   '#E74C3C'   # Pure red
 ]

# Another option with even more contrast
# colors_4 = [
#     '#4C8BF5',  # Bright blue
#     '#3A73C4',  # Medium blue
#     '#C94F3C',  # Medium red
#     '#E74C3C'   # Bright red
# ]

for i, col in enumerate(cat_cols):
    churn_rate = (df_raw.groupby(col)['Churn']
                  .apply(lambda x: (x == 'Yes').mean() * 100)
                  .reset_index(name='churn_rate'))
    counts = df_raw[col].value_counts().reset_index()
    counts.columns = [col, 'count']
    merged = churn_rate.merge(counts, on=col)
    
    # Sort by churn rate for better visualization
    merged = merged.sort_values('churn_rate', ascending=False)
    
    # Create bars using matplotlib directly
    x_positions = range(len(merged))
    bars = axes[i].bar(x_positions, merged['churn_rate'])
    
    # Color bars based on churn rate quartiles
    churn_values = merged['churn_rate'].values
    quartiles = np.percentile(churn_values, [25, 50, 75])
    
    for bar, churn_val in zip(bars, churn_values):
        if churn_val <= quartiles[0]:
            bar.set_color(colors_4[0])  # Lowest churn - Blue
        elif churn_val <= quartiles[1]:
            bar.set_color(colors_4[1])  # Low-medium churn - Darker blue
        elif churn_val <= quartiles[2]:
            bar.set_color(colors_4[2])  # Medium-high churn - Darker red
        else:
            bar.set_color(colors_4[3])  # Highest churn - Red
        bar.set_edgecolor('white')
        bar.set_linewidth(1)  # Slightly thicker white lines for separation
    
    # Set x-tick labels
    axes[i].set_xticks(x_positions)
    axes[i].set_xticklabels(merged[col], rotation=20, ha='right', fontsize=8)
    
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Tasa de Churn (%)')
    axes[i].set_xlabel('')
    axes[i].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
    
    # Add value labels on top of bars for better readability
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            axes[i].text(bar.get_x() + bar.get_width()/2., height + 1,
                        f'{height:.0f}%', ha='center', va='bottom', fontsize=7, fontweight='bold')

# Ocultar subplots vacíos
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Tasa de Churn (%) por variable categórica', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Interpretación de hallazgos clave:**

| Variable | Observación |
|---|---|
| `Contract` | Los clientes `Month-to-month` tienen tasa de churn ~43%, muy superior a contratos anuales (~11%) o bienales (~3%). **Variable de mayor poder predictivo**. |
| `InternetService` | Clientes con `Fiber optic` abandonan más (~42%) que los de `DSL` (~19%). |
| `OnlineSecurity` / `TechSupport` | Quienes no tienen estos servicios presentan tasas de churn casi el doble. |
| `PaymentMethod` | El método `Electronic check` se asocia con mayor churn (~45%). |
| `PaperlessBilling` | Los clientes con factura electrónica tienen mayor tasa de abandono. |
| `gender` | Prácticamente sin diferencia entre géneros — baja relevancia predictiva. |

## 6. Calidad de datos y valores faltantes

In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw)) * 100

missing_df = pd.DataFrame({
    'Valores faltantes': missing,
    'Porcentaje (%)': missing_pct.round(3)
}).query('`Valores faltantes` > 0')

if missing_df.empty:
    print('No se detectaron valores NaN directos en el dataset.')
else:
    print(missing_df)

In [ ]:
# TotalCharges tenía espacios en blanco en lugar de valores numéricos
# Al convertir con errors='coerce' esos se vuelven NaN
n_nan = df_raw['TotalCharges'].isnull().sum()
print(f'Valores NaN en TotalCharges tras conversión: {n_nan}')
print()
# Perfil de esos registros
print('Perfil de registros con TotalCharges nulo:')
print(df_raw[df_raw['TotalCharges'].isnull()][['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].head(12))

**Interpretación:**  
Los 11 registros con `TotalCharges` nulo corresponden a clientes con `tenure = 0`, es decir, que aún no han completado su primer mes de servicio. Su `TotalCharges` lógicamente es 0.0 (o nulo porque el sistema no registró el primer cobro aún). Se imputarán con 0.0 como estrategia conservadora.

## 7. Limpieza y preprocesamiento

### 7.1 Tratamiento de valores faltantes y tipo de datos

In [ ]:
df = df_raw.copy()

# 1. TotalCharges: convertir y rellenar NaN con 0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0.0)

# 2. Eliminar columna customerID (identificador sin valor predictivo)
df.drop(columns=['customerID'], inplace=True)

# 3. Variable objetivo: codificar Churn como binario (Yes=1, No=0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f'Shape tras limpieza: {df.shape}')
print(f'Valores nulos restantes: {df.isnull().sum().sum()}')
df.head(3)

### 7.2 Identificación de tipos de columnas

In [ ]:
TARGET = 'Churn'

numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

# SeniorCitizen es numérica (0/1) pero conceptualmente binaria
# Se incluye en numéricas para escalar junto con las demás
numerical_features_all = ['SeniorCitizen'] + numerical_features

categorical_features = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
]

print('Numéricas:', numerical_features_all)
print('\nCategoricas:', categorical_features)
print(f'\nTotal features: {len(numerical_features_all) + len(categorical_features)}')

### 7.3 Separación en X e y, y split train/test

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Train: {X_train.shape[0]:,} registros  |  Test: {X_test.shape[0]:,} registros')
print(f'\nProporción de Churn en train: {y_train.mean():.3f}')
print(f'Proporción de Churn en test:  {y_test.mean():.3f}')
print('\nEl uso de stratify=y garantiza que ambos splits mantengan la misma proporción de churn.')

## 8. Construcción del Pipeline de preprocesamiento

Se diseña un `ColumnTransformer` que aplica transformaciones diferenciadas según el tipo de variable, listo para ser integrado en los pipelines de modelado del Notebook 2.

In [ ]:
# Pipeline para variables numéricas:
# 1. Imputación con la mediana (robusta a outliers)
# 2. Escalado estándar (media 0, std 1)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para variables categóricas:
# 1. Imputación con el valor más frecuente
# 2. OneHotEncoding con drop='first' para evitar multicolinealidad perfecta
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer: combina ambos pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features_all),
        ('cat', categorical_transformer, categorical_features)
    ]
)

### 8.1 Verificación: fit_transform sobre el conjunto de entrenamiento

In [ ]:
# Ajustar SOLO sobre train para evitar data leakage
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed  = preprocessor.transform(X_test)

print(f'Shape X_train transformado: {X_train_transformed.shape}')
print(f'Shape X_test transformado:  {X_test_transformed.shape}')

# Recuperar nombres de las columnas resultantes
ohe_cols = (preprocessor
            .named_transformers_['cat']['onehot']
            .get_feature_names_out(categorical_features)
            .tolist())

all_feature_names = numerical_features_all + ohe_cols
print(f'\nTotal de features tras encoding: {len(all_feature_names)}')
print('\nPrimeras 10 features:', all_feature_names[:10])

**Interpretación:**  
El preprocesador expande las 19 columnas originales a ~30 features tras aplicar OHE (con `drop='first'` para eliminar una dummy por variable). El ajuste se realizó **exclusivamente sobre `X_train`** y luego se aplicó `transform` sobre `X_test`, evitando así fuga de información (*data leakage*).

## 9. Análisis adicional: outliers en variables numéricas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for i, col in enumerate(numerical_features):
    sns.boxplot(y=df[col], ax=axes[i], color='#5BA4CF',
                flierprops={'marker':'o', 'markersize':3, 'alpha':0.5})
    axes[i].set_title(f'Boxplot: {col}')
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    outliers = df[(df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)][col]
    axes[i].set_xlabel(f'Outliers IQR: {len(outliers)}')

plt.suptitle('Outliers en variables numéricas (criterio IQR)', fontsize=13)
plt.tight_layout()
plt.show()

**Interpretación:**  
No se detectan outliers extremos que requieran tratamiento especial. Los valores altos en `TotalCharges` corresponden a clientes de alta antigüedad y son legítimos. Los modelos basados en árboles (Random Forest, XGBoost, etc.) son **robustos a outliers** por naturaleza, por lo que no se aplicará ningún capping adicional.

## 10. Exportación de objetos para Notebook 2

In [ ]:
import joblib, os

os.makedirs('data', exist_ok=True)

# Guardar el dataset limpio
df.to_csv('data/telco_churn_clean.csv', index=False)

# Guardar el preprocessor ajustado (para uso en API y Notebook 2)
joblib.dump(preprocessor, 'data/preprocessor.joblib')

# Guardar los splits para garantizar reproducibilidad entre notebooks
joblib.dump((X_train, X_test, y_train, y_test), 'data/train_test_splits.joblib')

# Guardar las listas de features para referencia
joblib.dump({
    'numerical': numerical_features_all,
    'categorical': categorical_features,
    'all_encoded': all_feature_names
}, 'data/feature_lists.joblib')

print('Archivos exportados:')
for f in os.listdir('data'):
    size = os.path.getsize(f'data/{f}')
    print(f'  data/{f}  ({size:,} bytes)')

## 11. Resumen del EDA y decisiones de preprocesamiento

| Aspecto | Hallazgo | Decisión tomada |
|---|---|---|
| Valores faltantes | 11 NaN en `TotalCharges` (tenure=0) | Imputar con 0.0 |
| Tipo de dato | `TotalCharges` era string | Convertir con `pd.to_numeric` |
| Variable ID | `customerID` sin valor predictivo | Eliminar |
| Variable objetivo | `Churn` = Yes/No | Codificar como 1/0 |
| Desbalance | ~73% No / ~27% Yes | Tratar con `class_weight` o `scale_pos_weight` en Notebook 2 |
| Outliers | Ninguno problemático | Sin acción (árboles son robustos) |
| Variables numéricas | tenure, MonthlyCharges, TotalCharges, SeniorCitizen | Imputar mediana + StandardScaler |
| Variables categóricas | 15 columnas object | Imputar moda + OneHotEncoder (drop='first') |
| Split | 80% train / 20% test | `stratify=y` para preservar distribución de clases |

**Variable más discriminante según EDA:** `Contract` (Month-to-month vs anual/bienal), seguida de `InternetService` (Fiber optic), `OnlineSecurity` y `PaymentMethod` (Electronic check).